In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import os
from sklearn.metrics import confusion_matrix, roc_auc_score, accuracy_score, f1_score, recall_score, precision_score
import warnings

# Ignoramos warnings de división por cero (común en validación cruzada si una clase no aparece)
warnings.filterwarnings('ignore')

### CONFIGURACION



In [ ]:
# Algoritmos individuales y Ensembles
METHODS = [
    "KNN", "SVM", "NaiveBayes", "RandomForest", 
    "Ensemble_Votacion", "Ensemble_Media", "Ensemble_Mediana"
]

# Versiones del conjunto de datos
DATASETS = [
    "original", "estandarizado", "normalizado",
    "originalPCA95", "originalPCA80",
    "estandarizadoPCA95", "estandarizadoPCA80",
    "normalizadoPCA95", "normalizadoPCA80"
]

# Métricas solicitadas en el enunciado
METRICS = [
    "F1", "Sensibilidad", "Exactitud", "Especificidad", 
    "Recall", "Precision", "FNR", "FPR", "AUC"
]

FOLDS = 5  # k-fold CV


### CARGA Y AGREGACIÓN DE DATOS

In [ ]:
PATH_PREDICCIONES = "./predicciones/" # Ajustar si es necesario

def load_and_aggregate_results():
    aggregated_data = []
    print("Iniciando carga y cálculo de métricas...")

    for method in METHODS:
        # Ajuste del nombre del archivo según el método
        method_file_name = method.lower() if "Ensemble" not in method else method # Ajuste según tus nombres de archivo
        # Iterar por los datasets
        for dataset in DATASETS:
            fold_metrics = {m: [] for m in METRICS}
            
            # Iterar por los 5 folds
            for k in range(1, FOLDS + 1):
               
                filename = f"{PATH_PREDICCIONES}pred_{k}_{dataset}_{method_file_name}.csv"
                
                # Una vez leído el archivo, extraemos y calculamos métricas
                try:
                    df = pd.read_csv(filename)
                    
                    # Extraer vectores
                    y_true = df['y_true'].values
                    y_pred = df['y_pred'].values
                    
                    # Extraer probabilidades si existen
                    proba_cols = [c for c in df.columns if 'prob_' in c]
                    y_proba = df[proba_cols].values if proba_cols else None
                    
                    # Recalcular métricas
                    metrics_dict = calculate_custom_metrics(y_true, y_pred, y_proba)
                    
                    # Por cada métrica, agregar al fold correspondiente
                    for m in METRICS:
                        fold_metrics[m].append(metrics_dict[m])
                        
                except FileNotFoundError:
                    # Si no existe el archivo, saltamos
                    print(f"Falta: {filename}") 
                    pass

            # Solo agregamos si encontramos datos
            if len(fold_metrics["F1"]) > 0:
                row = {'Method': method, 'Dataset': dataset}
                for m in METRICS:
                    # Evitar warnings de media de lista vacía
                    values = fold_metrics[m] if fold_metrics[m] else [0]
                    
                    mean_val = np.mean(values)
                    std_val = np.std(values)
                    
                    row[f"{m}_mean"] = mean_val
                    row[f"{m}_std"] = std_val
                    
                    # Formato LaTeX para Tablas
                    row[f"{m}_display"] = f"{mean_val:.4f} $\\pm$ {std_val:.4f}"
                
                aggregated_data.append(row)

    print("Carga completada.")
    return pd.DataFrame(aggregated_data)

# Ejecutar
df_results = load_and_aggregate_results()
df_results.head() # Verificar que hay datos reales

### GENERACIÓN DE TABLAS PARA LATEX

In [ ]:
def generate_method_tables(df):
    """
    Genera una tabla para CADA método donde:
    Filas: Datasets
    Columnas: Métricas (formato texto con media y std)
    """
    for method in METHODS:
        print(f"--- Tabla Resumen para: {method} ---")
        subset = df[df['Method'] == method].copy()
        
        # Seleccionar solo las columnas de display (texto)
        cols_display = [f"{m}_display" for m in METRICS]
        table_display = subset.set_index('Dataset')[cols_display]
        
        # Renombrar columnas para quitar "_display"
        table_display.columns = METRICS
        
        # Imprimir formato compatible con Pandas (o exportar a LaTeX)
        print(table_display)
        print("\n" + "="*50 + "\n")
        
        # Para exportar a LaTeX directamente:
        # print(table_display.to_latex(escape=False))

def generate_f1_summary_table(df):
    """
    Tabla específica para F1-Score:
    Filas: Métodos
    Columnas: Datasets
    """
    print("--- Tabla Resumen F1-Score (Comparativa Global) ---")
    
    # Pivotar la tabla
    pivot_f1 = df.pivot(index='Method', columns='Dataset', values='F1_display')
    
    print(pivot_f1)
    # print(pivot_f1.to_latex(escape=False))

# Ejecutar generación
generate_method_tables(df_results)
generate_f1_summary_table(df_results)

### GENERACIÓN DE GRÁFICAS


In [ ]:
# Métodos para graficar comparaciones solicitadas en el enunciado FN vs FP, PR vs RC, ACC vs Fm
def plot_comparisons(df):
    # Mapeo de colores por método para distinguir en la gráfica
    colors = plt.cm.tab10(np.linspace(0, 1, len(METHODS)))
    method_color_map = dict(zip(METHODS, colors))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Pares de métricas a graficar (Eje X, Eje Y)
    plot_pairs = [
        ('FPR_mean', 'FNR_mean', 'FPR vs FNR (Simulando FP vs FN)'), # Ajustar según tus métricas exactas
        ('Recall_mean', 'Precision_mean', 'Precision vs Recall'),
        ('F1_mean', 'Exactitud_mean', 'Accuracy vs F1-Score')
    ]
    
    for ax, (x_metric, y_metric, title) in zip(axes, plot_pairs):
        for method in METHODS:
            subset = df[df['Method'] == method]
            ax.scatter(
                subset[x_metric], 
                subset[y_metric], 
                label=method, 
                color=method_color_map[method],
                alpha=0.7
            )
        
        ax.set_xlabel(x_metric.replace('_mean', ''))
        ax.set_ylabel(y_metric.replace('_mean', ''))
        ax.set_title(title)
        ax.grid(True, linestyle='--', alpha=0.5)

    # Leyenda única
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.05), ncol=len(METHODS))
    plt.tight_layout()
    plt.show()

# Ejecutar gráficas
plot_comparisons(df_results)